In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io
import matplotlib.pyplot as plt
import os
import anndata
import gc
from scipy.sparse import csr_matrix

seed = 42
np.random.seed(seed)

In [2]:
data_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/Ada/perturbation_modeling"

main_path = "/rds/general/user/ap5625/home/perturbation_modeling/perturbGEN/adipocytes_HVG"

broad_path = "/rds/general/user/ap5625/home/perturbation_modeling/"

In [3]:
adata = sc.read_h5ad(data_path + "/adipocytes_subset_hvg_only.h5ad")

In [6]:
adata.var_names_make_unique()
adata.obs_names_make_unique()

In [7]:
print(adata)

AnnData object with n_obs × n_vars = 155923 × 1969
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_0

In [10]:
print(adata.layers["raw"].max())

781.0


In [9]:
print(adata.layers["norm_log1p"].max())

7.1202602


In [11]:
adata.X = adata.layers["raw"].copy()

## ensemble Mapping

In [13]:
# map to ENSEMBL IDs

ref = pd.read_csv(broad_path + "/mart_export.txt", sep="\t")

print(ref.columns.tolist())

['Gene stable ID', 'Gene name']


In [14]:
symbol_to_ensembl = dict(zip(ref["Gene name"], ref["Gene stable ID"]))


def map_to_ensembl(gene_name):
    if str(gene_name).startswith("ENSG"):
        return gene_name
    else:
        return symbol_to_ensembl.get(gene_name, None)


adata.var["ensembl_id"] = [map_to_ensembl(g) for g in adata.var_names]

missing = adata.var["ensembl_id"].isna().sum()
print(f"Still unmatched: {missing}")
print(adata.var[adata.var["ensembl_id"].isna()].index.tolist()[:20])

adata = adata[:, adata.var["ensembl_id"].notna()].copy()
adata.var_names = adata.var["ensembl_id"].values
print(f"Final gene count: {adata.n_vars}")

Still unmatched: 10
['DIRC3-AS1', 'LINC00886', 'LINC02224', 'C5orf64', 'LINC01230', 'C12orf40', 'HIF1A-AS3', 'LINC02256-1', 'LINC02246', 'ODF3B']
Final gene count: 1959


In [15]:
ensembl_to_symbol = dict(zip(ref["Gene stable ID"], ref["Gene name"]))

adata.var["gene_symbol"] = adata.var_names.map(ensembl_to_symbol)

# ensembl_id column is already set as var_names but add it as a column too
adata.var["ensembl_id"] = adata.var_names

# Verify
print(adata.var[["ensembl_id", "gene_symbol"]].head(10))

                      ensembl_id gene_symbol
ENSG00000131584  ENSG00000131584       ACAP3
ENSG00000221978  ENSG00000221978       CCNL2
ENSG00000131697  ENSG00000131697       NPHP4
ENSG00000158292  ENSG00000158292      GPR153
ENSG00000215788  ENSG00000215788    TNFRSF25
ENSG00000171680  ENSG00000171680     PLEKHG5
ENSG00000116285  ENSG00000116285      ERRFI1
ENSG00000074800  ENSG00000074800        ENO1
ENSG00000142583  ENSG00000142583      SLC2A5
ENSG00000228526  ENSG00000228526    MIR34AHG


In [16]:
print(f"Total genes: {adata.n_vars}")
print(f"Genes with symbol: {adata.var['gene_symbol'].notna().sum()}")

Total genes: 1959
Genes with symbol: 1731


In [17]:
# 5. Set var_names to ensembl_id
adata.var_names = adata.var["ensembl_id"]

In [18]:
print(f"adata now has {adata.n_vars} genes with Ensembl IDs as var_names.")

adata now has 1959 genes with Ensembl IDs as var_names.


In [19]:
adata.var.index.name = "gene_id"

In [20]:
adata.var

,highly_variable,means,dispersions,dispersions_norm,highly_variable_nbatches,highly_variable_intersection,mean,std,ensembl_id,gene_symbol
gene_id,,,,,,,,,,
ENSG00000131584,True,0.050387,1.777334,0.580767,1,False,-2.891568e-12,0.262811,ENSG00000131584,ACAP3
ENSG00000221978,True,0.165230,1.746790,0.677606,1,False,2.511153e-13,0.625189,ENSG00000221978,CCNL2
ENSG00000131697,True,0.055837,1.820596,0.850299,2,True,2.043988e-11,0.281743,ENSG00000131697,NPHP4
ENSG00000158292,True,0.059743,1.773031,0.632851,1,False,2.010500e-11,0.264667,ENSG00000158292,GPR153
ENSG00000215788,True,0.102906,1.904987,1.120917,2,True,1.281070e-11,0.407655,ENSG00000215788,TNFRSF25
...,...,...,...,...,...,...,...,...,...,...
ENSG00000291033,True,0.078802,1.727515,0.501781,1,False,1.295615e-11,0.281207,ENSG00000291033,TXLNGY
ENSG00000229236,True,0.270394,2.053086,2.027001,2,True,-2.501064e-11,0.502827,ENSG00000229236,TTTY10
ENSG00000228253,True,0.020711,2.552146,2.144043,0,False,-4.949258e-12,0.097128,ENSG00000228253,MT-ATP8


## obs preprocessing

In [21]:
print(adata.obs.columns.tolist())

['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_0.5', 'leiden_1.0', 'Remission_2021', 'Age', 'Ethnicity', 'F

In [4]:
# only include obese and weight loss for the start - as they are paired


adata_sub = adata[
    adata.obs["condition"].isin(["baseline", "weightloss"])
].copy()

In [5]:
print(adata.obs["condition"].value_counts())
print(adata_sub.obs["condition"].value_counts())

condition
weightloss    91273
baseline      56955
Lean           7695
Name: count, dtype: int64
condition
weightloss    91273
baseline      56955
Name: count, dtype: int64


In [7]:
baseline_donors = set(
    adata_sub.obs[adata_sub.obs["condition"] == "baseline"]["Donor"].unique()
)
weight_loss_donors = set(
    adata_sub.obs[adata_sub.obs["condition"] == "weightloss"]["Donor"].unique()
)


paired = baseline_donors & weight_loss_donors

len(paired)

66

In [29]:
len(baseline_donors)

69

In [28]:
len(weight_loss_donors)

67

In [31]:
adata_sub.obs["donor_id"] = adata_sub.obs["Donor"]

In [32]:
adata_sub.obs["condition"] = adata_sub.obs["condition"].map(
    {"baseline": "obese", "weightloss": "weightloss"}
)

In [33]:
adata_sub.obs["cell_states_adipocytes"] = adata_sub.obs["cell_state_t2d"]

In [35]:
adata_sub.write(data_path + "/preprocessed_adipocytes_hvg_perturbgen.h5ad")